In [30]:
!pip install -q groq chromadb pypdf

In [31]:
from groq import Groq
from google.colab import userdata
api_key = userdata.get("FoodAPIKey")
client = Groq(api_key = api_key)
MODEL = "openai/gpt-oss-120b"

In [32]:
import os
os.makedirs("docs",exist_ok = True)

In [33]:
print(os.listdir("docs"))

['Healthy_Eating_Guide.pdf', '.ipynb_checkpoints']


In [41]:
import re
from pypdf import PdfReader

def load_text(path):
  with open(path, "r", encoding = "utf-8") as f:
    return[{"text":f.read(),"source": os.path.basename(path),"page":1}]

def load_pdf(path):
  pages = []
  for number,page in enumerate(PdfReader(path).pages, start = 1):
    pages.append({"text":page.extract_text() or "", "source":os.path.basename(path),"page":number})
  return pages

def load_folder(folder):
  pages = []
  for name in sorted(os.listdir(folder)):
    path = os.path.join(folder, name)
    if name.lower().endswith(".txt"):
      pages += load_text(path)
    elif name.lower().endswith(".pdf"):
      pages += load_pdf(path)
  return pages

In [42]:
def clean_text(text):
  return " ".join(text.split())

In [49]:
import os
def chunk_sentences(text, max_chars=400):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks = []
    current = ""
    for sentence in sentences:
        if current and len(current) + 1 + len(sentence) > max_chars:
            chunks.append(current)
            current = sentence
        else:
            current = (current + " " + sentence).strip()
    if current:
        chunks.append(current)
    return chunks

def chunk_paragraphs(text, max_chars=400):
    chunks = []
    for paragraph in text.split("\n\n"):
        paragraph = clean_text(paragraph)
        if paragraph:
            chunks += chunk_sentences(paragraph, max_chars)
    return chunks


def build_chunks(folder, max_chars = 400):
    all_chunks = []
    for page in load_folder(folder):
        if not page["text"].strip():
            print(f"Empty page skipped:", page["source"], "page", page["page"])
            continue
        for piece in chunk_paragraphs(page["text"], max_chars):
            all_chunks.append({"text": piece, "source": page["source"], "page": page["page"]})
    return all_chunks

chunks = build_chunks("/content/docs", max_chars=250)

print(len(chunks), "chunks")
print(chunks[10] if len(chunks) > 10 else chunks[0])

25 chunks
{'text': 'A healthy diet also means staying within your daily calorie needs.', 'source': 'Healthy_Eating_Guide.pdf', 'page': 1}


In [52]:
import chromadb
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

emb = DefaultEmbeddingFunction()
db = chromadb.Client()

def save_chunks(parts, name="foods"):
    try:
        db.delete_collection(name)
    except Exception:
        pass

    col = db.create_collection(name=name, embedding_function=emb)

    col.upsert(
        documents=[p["text"] for p in parts],
        metadatas=[
            {"source": p["source"], "page": p["page"]}
            for p in parts
        ],
        ids=[f"food_{i}" for i in range(len(parts))]
    )
    return col
collection = save_chunks(chunks)
print("Stored:", collection.count(), "food chunks")

Stored: 25 food chunks


In [56]:
def find_food(query, top_n=2):
    total = collection.count()
    if total == 0:
        return []

    result = collection.query(query_texts=[query], n_results=min(top_n, total))
    matches = []

    for text, meta, dist in zip(result["documents"][0], result["metadatas"][0], result["distances"][0]):
        matches.append({"text": text, "source": meta["source"], "page": meta["page"], "distance": dist})
    return matches

for item in find_food("What are the benefits of eating fruits and vegetables?"):
    print(f'{item["distance"]:.2f} | 'f'{item["source"]} page {item["page"]} | 'f'{item["text"][:90]}...')

0.73 | Healthy_Eating_Guide.pdf page 1 | Save Time + Money You probably already know that a healthy diet includes a variety of frui...
0.81 | Healthy_Eating_Guide.pdf page 1 | As part of a healthy diet,* eating fruits and vegetables instead of high-fat or high-calor...


In [59]:
FOOD_RULES = (
    "Use the supplied food-related passages to answer the user's question. "
    "Keep responses relevant to healthy eating, produce, meal ideas, "
    "nutrition, and saving money on food. Do not add facts from outside "
    "the passages. If the passages do not contain the answer, reply: "
    "I don't know about this information."
)

def make_food_answer(question, matches, chat_log):
    food_context = "\n\n".join(
        f'File: {item["source"]}, Page: {item["page"]}\n{item["text"]}'
        for item in matches
    )

    msgs = [{"role": "system", "content": FOOD_RULES}]
    msgs.extend(chat_log)
    msgs.append({
        "role": "user",
        "content": f"Reference material:\n{food_context}\n\nUser question: {question}"
    })

    result = client.chat.completions.create(
        model=MODEL,
        messages=msgs,
        temperature=0,
        max_tokens=400
    )
    return result.choices[0].message.content

chat_log = []
def ask_food_bot(question):
    matches = find_food(question, top_n=2)
    reply = make_food_answer(question, matches, chat_log[-6:])

    chat_log.append({"role": "user", "content": question})
    chat_log.append({"role": "assistant", "content": reply})

    return reply, matches


def display_reply(question):
    reply, matches = ask_food_bot(question)

    print("Question:", question)
    print("Response:", reply)

    refs = ", ".join(
        f'{item["source"]} (page {item["page"]})'
        for item in matches
    )
    print("References:", refs or "No references found")
    print()


while True:
    user_input = input("Ask a food question or exit: ")

    if user_input.strip().lower() == "exit":
        print("Chat ended.")
        break

    display_reply(user_input)

Ask a food question or exit: what is pakistan flag colours
Question: what is pakistan flag colours
Response: I don't know about this information.
References: Healthy_Eating_Guide.pdf (page 2), Healthy_Eating_Guide.pdf (page 2)

Ask a food question or exit: Why is a healthy diet important?
Question: Why is a healthy diet important?
Response: A healthy diet is important because it gives you a wide variety of fruits and vegetables that supply essential vitamins and minerals. Those foods are generally lower in calories and higher in fiber, which helps you stay within your daily calorie needs and supports overall health.
References: Healthy_Eating_Guide.pdf (page 1), Healthy_Eating_Guide.pdf (page 1)

Ask a food question or exit: What does the guide say about nutrition?
Question: What does the guide say about nutrition?
Response: The guide notes two key points about nutrition:

1. **Calorie control** – A healthy diet means staying within your daily calorie needs. (Page 1)

2. **Detailed nut